# 🧩 Agent Skills in Microsoft Foundry (Preview)

Welcome to the **Agent Skills** tutorial! As agents grow beyond prototypes, teams accumulate behavioral guidelines that must stay consistent across every conversation — a compliance disclaimer, a loan-review checklist, an escalation policy. Embedding these in each agent's system prompt creates duplication: change the policy and you must update and redeploy every agent.

**Skills** solve this by decoupling behavioral guidelines from agent code. A skill is a `SKILL.md` file that you author once and store centrally in Foundry through the versioned **Skills API**.

> ⚠️ **Preview.** Skills are in public preview. This preview is provided without an SLA and isn't recommended for production. Requires a recent `azure-ai-projects` SDK and `AIProjectClient(..., allow_preview=True)`.

### What you'll do

1. **Author** a `SKILL.md` for a banking **loan-review checklist**.
2. **Upload** it as a versioned skill (inline content and from a file).
3. **Version** it — add a v2 and switch the `default_version`.
4. **Attach** the skill to a **Toolbox** so any MCP client can discover it.
5. **Download** and **manage** (list, get, delete) skills.

### Two delivery modes

| Mode | How it works | When to use |
| --- | --- | --- |
| **Attach to a Toolbox** | Skills and tools share one MCP endpoint; clients discover them via [MCP Resources](https://modelcontextprotocol.io/docs/concepts/resources) (`resources/list` → `resources/read`). | Reuse the same skill across many agents / MCP clients. |
| **Direct injection (hosted agent)** | Download `SKILL.md` into the agent project; the runtime injects it as extra instructions each session. | Bundle a specific skill version with your agent code. |

### 🧠 Skills vs. Tools

> **Tools** define **what** an agent can do (call an API, search an index). **Skills** define **how** it performs a task (the reusable, versioned instructions/workflow). This notebook focuses on skills.

## Prerequisites

- A Microsoft Foundry project with the **Foundry User** role.
- A `.env` file in the parent directory containing:
  ```bash
  TENANT_ID=<your-azure-tenant-id>
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=<supported-model>
  ```

📚 **Learn more:** [Use skills with Foundry agents](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/skills) · [Toolbox overview](https://learn.microsoft.com/azure/foundry/agents/concepts/toolbox-overview) · [Private skill catalog](https://learn.microsoft.com/azure/foundry/agents/how-to/private-skill-catalog)


## 🔐 Authentication Setup

Before running the next cell, sign in with Azure CLI so `AzureCliCredential` can obtain tokens:

```bash
az login --use-device-code
```

The Skills API authenticates with a bearer token (scope `https://ai.azure.com/.default`) and requires the preview feature header, which the SDK adds automatically when you pass `allow_preview=True`.


## 1. Initial Setup

Load environment variables and initialize an **AIProjectClient** with `allow_preview=True` (required for the preview Skills API surface `project.beta.skills`).


In [4]:
import os
import tempfile
from datetime import datetime
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import SkillInlineContent
from azure.identity import AzureCliCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")
load_dotenv(dotenv_path)

tenant_id = os.getenv("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

required_settings = {
    "TENANT_ID": tenant_id,
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_deployment,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

credential = AzureCliCredential(tenant_id=tenant_id)

# allow_preview=True is required to access the preview Skills API (project.beta.skills).
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    allow_preview=True,
)
print("Initialized AIProjectClient with preview features enabled (Skills API ready).")


Initialized AIProjectClient with preview features enabled (Skills API ready).


## 2. Author a Skill 📝

A skill follows the [Agent Skills](https://agentskills.io) format: a Markdown file with a small YAML front matter block. The body becomes the skill's injected instructions.

```markdown
---
name: loan-review-checklist
description: Standard checklist and disclaimers a loan advisor must apply to every review.
---

# Loan Review Checklist
...instructions...
```

**Front matter rules** (from the docs):

| Field | Required | Rules |
| --- | --- | --- |
| `name` | Yes | URL path key. Lowercase letters, numbers, hyphens only. No leading/trailing or consecutive hyphens. Max 64 chars. Pattern `^[a-z0-9]([a-z0-9\-]*[a-z0-9])?$`. **Unquoted.** |
| `description` | Yes | One-liner shown in listings. Max 1,024 chars. **Unquoted.** |
| Body | Yes | Free Markdown → the skill's injected instructions. |

Below we author the checklist content our loan advisor agent should apply on every turn.


In [5]:
SKILL_NAME = "loan-review-checklist"

SKILL_DESCRIPTION = (
    "Standard checklist and disclaimers a loan advisor must apply to every review."
)

# The instructions body — the reusable behavioral guideline shared across agents.
SKILL_INSTRUCTIONS = """You are applying the bank's standard Loan Review Checklist.

For every loan-related answer:
1. State that you are not a licensed financial advisor and cannot give personalized advice.
2. Explain relevant terms plainly (APR, DTI, credit score, amortization) before using them.
3. Walk through the checklist in order:
   - Verify the requested amount against stated income (flag if amount > 5x annual income).
   - Comment on how credit score bands (<620, 620-699, 700-759, 760+) affect rate and approval.
   - Note debt-to-income (DTI) guidance: <=36% preferred, 37-43% cautionary, >43% high risk.
   - List typical closing costs the borrower should budget for.
4. Never encourage risky borrowing; always recommend consulting a licensed advisor.
5. End with a one-line regulatory disclaimer.
"""

print(f"Authored skill '{SKILL_NAME}' ({len(SKILL_INSTRUCTIONS)} chars of instructions).")


Authored skill 'loan-review-checklist' (791 chars of instructions).


## 3. Create a Skill Version (inline content)

Creating a version **auto-creates the parent skill** if it doesn't exist. Each version is an immutable `SkillVersion`; the parent `Skill` tracks `default_version` and `latest_version`. The simplest path submits the content directly with `SkillInlineContent`.


In [6]:
# Create skill version 1 from inline content (auto-creates the parent skill).
created = project_client.beta.skills.create(
    name=SKILL_NAME,
    inline_content=SkillInlineContent(
        description=SKILL_DESCRIPTION,
        instructions=SKILL_INSTRUCTIONS,
    ),
)
print(f"Created skill: {created.name}  version: {created.version}")


Created skill: loan-review-checklist  version: 1


## 4. Create a Version from a `SKILL.md` File (optional)

You can also upload a `SKILL.md` (or a ZIP of a skill folder plus assets). The server parses the front matter to populate the version's `description` and `instructions`. Here we write a real `SKILL.md` to disk and upload it as **version 2**.

> Skill names in the file's front matter must match the skill name you upload to.


In [7]:
from azure.ai.projects.models import CreateSkillVersionFromFilesBody

# Version 2 adds an explicit escalation rule for high-risk applications.
skill_md = f"""---
name: {SKILL_NAME}
description: {SKILL_DESCRIPTION}
---

# Loan Review Checklist (v2)

{SKILL_INSTRUCTIONS}
## Escalation
- If DTI > 43% or requested amount > 5x annual income, recommend a human underwriter review
  before proceeding, and do not imply approval.
"""

skill_dir = Path(tempfile.gettempdir()) / SKILL_NAME
skill_dir.mkdir(parents=True, exist_ok=True)
skill_md_path = skill_dir / "SKILL.md"
skill_md_path.write_text(skill_md, encoding="utf-8")

imported = project_client.beta.skills.create_from_files(
    SKILL_NAME,
    content=CreateSkillVersionFromFilesBody(
        files=[(skill_md_path.name, skill_md_path.read_bytes())]
    ),
)
print(f"Uploaded {skill_md_path} -> {imported.name} version: {imported.version}")


Uploaded C:\Users\GABHAR~1\AppData\Local\Temp\loan-review-checklist\SKILL.md -> loan-review-checklist version: 2


## 5. Version Management: List, Inspect, and Promote

Versions are immutable. To "update" a skill you create a new version, test it, then **promote** it by pointing `default_version` at it. Toolboxes and agents that reference the skill without pinning a version automatically pick up the new default — no agent code changes.


In [8]:
# List every skill in the project.
skills = list(project_client.beta.skills.list())
print(f"Found {len(skills)} skill(s):")
for skill in skills:
    print(f"  {skill.name} (default: {skill.default_version})")

# List the versions of our skill.
versions = list(project_client.beta.skills.list_versions(SKILL_NAME))
print(f"\n'{SKILL_NAME}' has {len(versions)} version(s): "
      f"{', '.join(v.version for v in versions)}")

# Promote version 2 to be the default (metadata-only repoint).
updated = project_client.beta.skills.update(SKILL_NAME, default_version="2")
print(f"Default version is now: {updated.default_version}")


Found 1 skill(s):
  loan-review-checklist (default: 1)

'loan-review-checklist' has 2 version(s): 2, 1
Default version is now: 2


## 6. Attach the Skill to a Toolbox 🧰

A **Toolbox** exposes tools *and* skills through a single MCP-compatible endpoint. When you attach a skill, MCP clients (GitHub Copilot, Claude Code, Agent Framework, your own harness) discover it as an [MCP Resource](https://modelcontextprotocol.io/docs/concepts/resources): they call `resources/list` at startup and `resources/read` to load the body **only when needed** — the [Agent Skills](https://agentskills.io) progressive-disclosure pattern that keeps token usage low.

> Skills attached to a toolbox must live in the **same Foundry project**. Omit `version` to follow the skill's `default_version`, or pin a `version` string for an immutable reference.


In [9]:
from azure.ai.projects.models import ToolboxSkillReference

TOOLBOX_NAME = "advisor-toolbox"

# Create a toolbox version that references the skill (follows its default_version).
toolbox_version = project_client.toolboxes.create_version(
    name=TOOLBOX_NAME,
    description="Toolbox exposing the loan-review checklist skill",
    tools=[],
    skills=[ToolboxSkillReference(name=SKILL_NAME)],  # add version="1" to pin
)
print(f"Created toolbox '{TOOLBOX_NAME}' version: {toolbox_version.version}")

# Any MCP client can consume the skill from this endpoint via MCP Resources.
toolbox_mcp_url = (
    f"{project_endpoint}/toolboxes/{TOOLBOX_NAME}"
    f"/versions/{toolbox_version.version}/mcp?api-version=v1"
)
print(f"Toolbox MCP endpoint:\n  {toolbox_mcp_url}")


Created toolbox 'advisor-toolbox' version: 1
Toolbox MCP endpoint:
  https://demopocaifoundry.services.ai.azure.com/api/projects/demoproject/toolboxes/advisor-toolbox/versions/1/mcp?api-version=v1


## 7. Download Skill Content

Download the active version as a ZIP archive — this is how a **hosted agent** pulls a skill for direct injection (extract `SKILL.md` into `skills/<name>/SKILL.md` and the runtime injects it each session).


In [11]:
download_folder = Path(tempfile.gettempdir()).resolve()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
download_path = download_folder / f"{SKILL_NAME}_{timestamp}.zip"

# download() returns an Iterator[bytes] for the skill's default version
# (use download_version(name, version) for a specific version).
download_path.write_bytes(b"".join(project_client.beta.skills.download(SKILL_NAME)))
print(f"Downloaded skill package ({download_path.stat().st_size} bytes) to:\n  {download_path}")


Downloaded skill package (754 bytes) to:
  C:\Users\gabhardwaj\AppData\Local\Temp\loan-review-checklist_20260921_204324.zip


## 8. Use the Skill in an Agent 🤖

Now prove the skill actually changes agent behavior. This is the **direct-injection** delivery mode: we pull the skill's `SKILL.md` **from Foundry** (the ZIP we just downloaded), strip its YAML front matter to get the instructions body, and use it as a prompt agent's instructions. The agent then applies the loan-review checklist on every turn — without the checklist text living in our notebook.

> This mirrors how a **hosted agent** consumes a skill: download from the Skills API, then inject the content as extra system instructions. (The other mode — Toolbox/MCP Resources discovery — is what you attached in Section 6.)


In [12]:
import zipfile
from io import BytesIO

from azure.ai.projects.models import PromptAgentDefinition


def extract_skill_body(zip_bytes: bytes) -> str:
    """Return the Markdown body of SKILL.md, stripping the YAML front matter."""
    with zipfile.ZipFile(BytesIO(zip_bytes)) as archive:
        skill_md_name = next(n for n in archive.namelist() if n.endswith("SKILL.md"))
        text = archive.read(skill_md_name).decode("utf-8")
    if text.startswith("---"):
        # Drop the front matter block between the first two '---' fences.
        text = text.split("---", 2)[-1]
    return text.strip()


# Reuse the ZIP downloaded in Section 7 and turn the skill into agent instructions.
skill_instructions = extract_skill_body(download_path.read_bytes())
print("Injecting skill instructions into the agent:\n")
print(skill_instructions[:400] + ("..." if len(skill_instructions) > 400 else ""))

skilled_agent = project_client.agents.create_version(
    agent_name="loan-advisor-skilled",
    description="Loan advisor that applies the loan-review-checklist skill",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=skill_instructions,
    ),
)
print(f"\nCreated agent: {skilled_agent.name} (version: {skilled_agent.version})")


Injecting skill instructions into the agent:

# Loan Review Checklist (v2)

You are applying the bank's standard Loan Review Checklist.

For every loan-related answer:
1. State that you are not a licensed financial advisor and cannot give personalized advice.
2. Explain relevant terms plainly (APR, DTI, credit score, amortization) before using them.
3. Walk through the checklist in order:
   - Verify the requested amount against state...

Created agent: loan-advisor-skilled (version: 1)


In [13]:
# Ask a loan question and watch the agent apply the injected checklist.
openai_client = project_client.get_openai_client()
skill_conversation = openai_client.conversations.create()

question = (
    "I earn $85,000 a year and want to borrow $500,000 for a home. "
    "My credit score is 640. What should I consider?"
)
print(f"User: {question}\n")

response = openai_client.responses.create(
    conversation=skill_conversation.id,
    input=question,
    extra_body={
        "agent_reference": {
            "type": "agent_reference",
            "name": skilled_agent.name,
            "version": skilled_agent.version,
        }
    },
)
if not response.output_text:
    raise RuntimeError("The skilled agent returned no text output.")

print("Agent (applying loan-review-checklist skill):\n" + "-" * 55)
print(response.output_text)


User: I earn $85,000 a year and want to borrow $500,000 for a home. My credit score is 640. What should I consider?

Agent (applying loan-review-checklist skill):
-------------------------------------------------------
I am not a licensed financial advisor and cannot give personalized advice. Instead, I’ll help you review your loan request using standard bank criteria.

**Relevant Terms Explained:**
- **APR (Annual Percentage Rate):** The total yearly cost of borrowing, including interest and fees, expressed as a percentage.
- **DTI (Debt-to-Income Ratio):** Your total monthly debt payments divided by your gross monthly income, used to assess if you can handle more debt.
- **Credit Score:** A number that shows your creditworthiness; higher scores mean better chances for loan approval and lower rates.
- **Amortization:** Paying off a loan with regular payments of principal and interest over time.

**Loan Review Checklist:**

1. **Requested Amount vs. Stated Income**
   - You are asking 

## 9. Cleanup

Delete the demo agent and conversation, delete the skill (removes all its versions), and close the client. To delete a single version instead, use `project_client.beta.skills.delete_version(SKILL_NAME, "1")`.

> A version can't be deleted while it's the `default_version` or referenced by a toolbox/agent. Repoint the default or remove references first.


In [ ]:
cleanup_errors = []

# Delete the demo agent and its conversation first (they reference the skill's behavior).
try:
    openai_client.conversations.delete(skill_conversation.id)
    print(f"Deleted conversation {skill_conversation.id}")
except Exception as error:
    cleanup_errors.append(f"conversation {skill_conversation.id}: {error}")

try:
    project_client.agents.delete_version(
        agent_name=skilled_agent.name,
        agent_version=skilled_agent.version,
    )
    print(f"Deleted {skilled_agent.name} version {skilled_agent.version}")
except Exception as error:
    cleanup_errors.append(f"{skilled_agent.name} version {skilled_agent.version}: {error}")

try:
    deleted = project_client.beta.skills.delete(name=SKILL_NAME)
    print(f"Deleted skill '{SKILL_NAME}': {deleted}")
except Exception as error:
    cleanup_errors.append(f"skill {SKILL_NAME}: {error}")

project_client.close()
credential.close()

if cleanup_errors:
    raise RuntimeError("Cleanup failures:\n" + "\n".join(cleanup_errors))
print("Skills notebook completed.")


## 📚 Summary

You built the full **Skills** lifecycle in Microsoft Foundry:

| Step | API |
| --- | --- |
| Create version (inline) | `project.beta.skills.create(name, inline_content=SkillInlineContent(...))` |
| Create version (file/ZIP) | `project.beta.skills.create_from_files(name, content=CreateSkillVersionFromFilesBody(...))` |
| List / get | `project.beta.skills.list()` · `list_versions(name)` |
| Promote default | `project.beta.skills.update(name, default_version="2")` |
| Attach to toolbox | `project.toolboxes.create_version(..., skills=[ToolboxSkillReference(name)])` |
| Download | `project.beta.skills.download(name)` · `download_version(name, version)` |
| Delete | `project.beta.skills.delete(name)` · `delete_version(name, version)` |

### Key concepts

1. **Skills decouple behavioral guidelines from agent code** — author once, reuse everywhere.
2. **Immutable, versioned** — every change is a new version; promote by repointing `default_version`.
3. **Two delivery modes** — attach to a **Toolbox** (MCP Resources discovery) or **download** for hosted-agent direct injection.
4. **Skills vs. tools** — tools = *what*; skills = *how*.

### Next steps

- **Govern at scale** with a [private skill catalog](https://learn.microsoft.com/azure/foundry/agents/how-to/private-skill-catalog) backed by **Azure API Center** — register org-scoped skills, define each skill's **allowed tools** (its governance boundary), and grant developers the *Azure API Center Data Reader* role to discover them under **Build → Tools → Skills → Browse skills**.
- Combine skills with the **MCP tool** — see [11-custom-mcp-server.ipynb](11-custom-mcp-server.ipynb).
- Bundle a skill into a **hosted agent** — see the `hosted-agents/` sample.
